**Important Links:**

* [Kaggle Notebook](https://www.kaggle.com/code/daemon49/flood-segmentation-model)
* [Test the Model Online](https://prayagxplorer.github.io/FloodGateAI/)

# UNet Training for Flood Segmentation

## Introduction to Semantic Segmentation
Semantic segmentation is the process of classifying every pixel in an image into a specific class. In this project, we are building a **binary semantic segmentation** model. This means our model looks at an aerial or satellite image and classifies every single pixel as either `1` (Flood Water) or `0` (Background).

This notebook walks through the entire pipeline: loading the dataset, applying data augmentation, building a PyTorch UNet model, training it with a combined BCE+Dice loss, and visualizing the predictions.

The code is written for local execution, automatically utilizing a GPU if available or falling back to CPU. 


## 1. Environment Setup

We rely on a few key libraries:
- **PyTorch**: For building and training the deep learning model.
- **Albumentations**: An extremely fast image augmentation library. Augmentation artificially expands our dataset by rotating, flipping, and altering the brightness of images, preventing the model from overfitting to the limited training data.
- **OpenCV & PIL**: For image loading and manipulation.

Ensure all dependencies are installed via the provided `requirements.txt`.


## 2. Dataset Discovery and Train/Validation Split

Before training, we must locate our dataset and split it into two sets:
- **Training Set (80%)**: Used by the model to learn the features of flood water.
- **Validation Set (20%)**: Used to evaluate the model on data it has never seen before. We use a fixed `random_state=42` to ensure our split is reproducible across multiple runs.

The code below will automatically locate the `dataset/` directory in your project folder and organize the images and masks into a `working/` directory for fast loading.


In [2]:
import os
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

import glob


DATASET_DIR = "./dataset"
if not os.path.exists(os.path.join(DATASET_DIR, "metadata.csv")):
    raise FileNotFoundError("Dataset not found in ./dataset. Please follow dataset/README.md instructions.")


WORKING_DIR = "./working"

IMAGE_DIR = os.path.join(DATASET_DIR, 'Image')
MASK_DIR = os.path.join(DATASET_DIR, 'Mask')
CSV_PATH = os.path.join(DATASET_DIR, 'metadata.csv')

# Load the image/mask filename mapping from metadata.csv.
# Read the CSV containing Image and Mask filenames
df = pd.read_csv(CSV_PATH)

# Split once with a fixed seed so experiments are repeatable.
# Split 80% for training and 20% for validation using a fixed random seed
train_df, valid_df = train_test_split(df, test_size=0.2, random_state=42)

def create_and_copy(dataframe, subset_name):
    img_out = os.path.join(WORKING_DIR, subset_name, 'Image')
    mask_out = os.path.join(WORKING_DIR, subset_name, 'Mask')
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(mask_out, exist_ok=True)
    
    for idx, row in tqdm(dataframe.iterrows(), total=len(dataframe), desc=f"Copying {subset_name}"):
        img_src = os.path.join(IMAGE_DIR, row['Image'])
        mask_src = os.path.join(MASK_DIR, row['Mask'])
        
        img_dst = os.path.join(img_out, row['Image'])
        mask_dst = os.path.join(mask_out, row['Mask'])
        
        if os.path.exists(img_src) and os.path.exists(mask_src):
            shutil.copy(img_src, img_dst)
            shutil.copy(mask_src, mask_dst)
        else:
            print(f"Warning: Missing file {row['Image']} or {row['Mask']}")

# Copy files into a consistent working layout used by the Dataset class.
print("Splitting data into train and valid folders...")
create_and_copy(train_df, 'train')
create_and_copy(valid_df, 'valid')
print("Data splitting complete.")

## 3. Dataloader, Augmentation, and Model Architecture

### The UNet Architecture
We use a **UNet** architecture, which is the industry standard for biomedical and semantic segmentation. It consists of three parts:
1. **Encoder (Contracting Path)**: A series of convolutional and max-pooling layers that extract deep features from the image while reducing its spatial dimensions.
2. **Bottleneck**: The deepest layer of the network where the most abstract features are represented.
3. **Decoder (Expanding Path)**: A series of upsampling (transpose convolution) layers that reconstruct the image's spatial resolution. Crucially, it uses **Skip Connections** to concatenate high-resolution features from the Encoder directly into the Decoder. This allows the model to precisely localize objects (flood boundaries) using fine-grained details.

### Loss Function (BCE + Dice)
Flood images suffer from extreme **class imbalance** (there are far more "normal" land pixels than "flood" pixels). 
- **Binary Cross Entropy (BCE)**: Evaluates pixel-by-pixel classification accuracy.
- **Dice Loss**: Evaluates the spatial overlap between the prediction and ground truth. It heavily penalizes the model if it fails to predict the minority class (the flood water).


In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np

import albumentations as A
from albumentations.pytorch import ToTensorV2

# --- Augmentation transforms (albumentations) ---
# Training gets random transforms; validation only gets deterministic preprocessing.
# ImageNet normalization constants
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

train_transform = A.Compose([
    A.Resize(256, 256),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Affine(translate_percent=0.05, scale=(0.9, 1.1), rotate=(-15, 15), p=0.4, border_mode=0),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
    A.CoarseDropout(num_holes_range=(2, 5), hole_height_range=(12, 20), hole_width_range=(12, 20), fill=0, p=0.2),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])

valid_transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])

# Custom Dataset returns one image tensor and one binary mask tensor per sample.
class SegmentationDataset(Dataset):
    def __init__(self, df, subset_name, transform=None):
        self.df = df.reset_index(drop=True)
        self.subset_name = subset_name
        self.transform = transform
        self.img_dir = os.path.join(WORKING_DIR, subset_name, 'Image')
        self.mask_dir = os.path.join(WORKING_DIR, subset_name, 'Mask')
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        img_name = self.df.loc[idx, 'Image']
        mask_name = self.df.loc[idx, 'Mask']
        
        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, mask_name)
        
        image = np.array(Image.open(img_path).convert("RGB").resize((256, 256), Image.BILINEAR))
        mask = np.array(Image.open(mask_path).convert("L").resize((256, 256), Image.NEAREST))
        
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']  # float32 tensor (C, H, W)
            mask = augmented['mask']    # uint8 tensor (H, W)
        
        # Cast mask to float and add channel dim; binarize
        mask = mask.float().unsqueeze(0)  # (1, H, W)
        mask = (mask > 127.0).float()     # binarize (0 or 1)
        
        return image, mask

train_dataset = SegmentationDataset(train_df, 'train', transform=train_transform)
valid_dataset = SegmentationDataset(valid_df, 'valid', transform=valid_transform)

# Using num_workers=4 for faster data loading
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train samples: {len(train_dataset)}, Valid samples: {len(valid_dataset)}")
print(f"Train batches: {len(train_loader)}, Valid batches: {len(valid_loader)}")

# --- UNet Model ---
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        # ENCODER PASS: Save outputs for skip connections
        return self.conv(x)

class UNet(nn.Module):
    """
    UNet architecture consisting of an Encoder, Bottleneck, and Decoder with skip connections.
    """
    def __init__(self, in_channels=3, out_channels=1, dropout_rate=0.15):
        super().__init__()
        self.down1 = DoubleConv(in_channels, 64)
        self.down2 = DoubleConv(64, 128)
        self.down3 = DoubleConv(128, 256)
        self.down4 = DoubleConv(256, 512)
        
        self.pool = nn.MaxPool2d(2)
        self.dropout = nn.Dropout2d(p=dropout_rate)
        
        self.up1 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.conv1 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv2 = DoubleConv(256, 128)
        self.up3 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv3 = DoubleConv(128, 64)
        
        self.out = nn.Conv2d(64, out_channels, 1)
        
    def forward(self, x):
        # ENCODER PASS: Save outputs for skip connections
        x1 = self.down1(x)
        p1 = self.pool(x1)
        
        x2 = self.down2(p1)
        p2 = self.pool(x2)
        
        x3 = self.down3(p2)
        p3 = self.pool(x3)
        
        x4 = self.down4(p3)
        x4 = self.dropout(x4)  # Dropout at bottleneck
        
        u1 = self.up1(x4)
        u1 = torch.cat([u1, x3], dim=1)
        c1 = self.conv1(u1)
        
        u2 = self.up2(c1)
        u2 = torch.cat([u2, x2], dim=1)
        c2 = self.conv2(u2)
        
        u3 = self.up3(c2)
        u3 = torch.cat([u3, x1], dim=1)
        c3 = self.conv3(u3)
        
        # Return RAW LOGITS — no sigmoid here
        return self.out(c3)

# Dice Coefficient (evaluation metric). Higher is better overlap.
def dice_coeff(logits, target, smooth=1e-6):
    """Compute Dice from raw logits. Applies sigmoid internally."""
    pred = (torch.sigmoid(logits) > 0.5).float()
    intersection = (pred * target).sum(dim=(2,3))
    union = pred.sum(dim=(2,3)) + target.sum(dim=(2,3))
    dice = (2. * intersection + smooth) / (union + smooth)
    return dice.mean().item()

# Differentiable Dice Loss (for training)
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        intersection = (probs * targets).sum(dim=(2,3))
        union = probs.sum(dim=(2,3)) + targets.sum(dim=(2,3))
        dice = (2. * intersection + self.smooth) / (union + self.smooth)
        return 1.0 - dice.mean()

# BCE handles pixel-level classification; Dice rewards mask overlap.
class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        return self.bce_weight * self.bce(logits, targets) + self.dice_weight * self.dice(logits, targets)

## 4. Training Loop

Here we train the model across multiple epochs. We utilize a few advanced training techniques:
- **Adam Optimizer**: An adaptive learning rate optimization algorithm.
- **Cosine Annealing**: A learning rate scheduler that starts with a high learning rate and smoothly decays it following a cosine curve. This helps the model settle into a precise local minima at the end of training.
- **Early Stopping**: We monitor the Validation Dice Score. If the score does not improve for a set number of consecutive epochs, we halt training early to prevent overfitting and save time.


In [4]:
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"Number of GPUs available: {torch.cuda.device_count()}")

model = UNet(in_channels=3, out_channels=1)

# Use DataParallel if multiple GPUs are available 
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs for training!")
    model = nn.DataParallel(model)

model = model.to(device)

# BCE handles pixel-level classification; Dice rewards mask overlap. — directly optimizes the Dice metric
criterion = BCEDiceLoss(bce_weight=0.5, dice_weight=0.5)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

num_epochs = 25

# Cosine Annealing: smoothly decays LR from 1e-3 toward 0 over all epochs
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

# Early stopping keeps the checkpoint with the strongest validation Dice.
best_valid_dice = 0.0
patience = 7
epochs_no_improve = 0

for epoch in range(num_epochs):
    model.train()
    train_dice = 0
    
    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]", leave=False):
        images = images.to(device)
        masks = masks.to(device)
        
        # 1. Clear previous gradients
        optimizer.zero_grad()
        # 2. Forward pass: compute predictions
        outputs = model(images)
        # 3. Calculate loss between predictions and ground truth
        loss = criterion(outputs, masks)
        
        # 4. Backward pass: compute gradients
        loss.backward()
        # 5. Update model weights
        optimizer.step()
        
        train_dice += dice_coeff(outputs, masks)
        
    # Set model to evaluation mode (disables dropout/batchnorm updates)
    model.eval()
    valid_dice = 0
    
    # Disable gradient calculation for validation to save memory and speed up computation
    with torch.no_grad():
        for images, masks in tqdm(valid_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Valid]", leave=False):
            images = images.to(device)
            masks = masks.to(device)
            
            # 2. Forward pass: compute predictions
        outputs = model(images)
            valid_dice += dice_coeff(outputs, masks)
    
    # Step the scheduler once per epoch after optimizer updates.
    scheduler.step()
            
    train_dice /= len(train_loader)
    valid_dice /= len(valid_loader)
    
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch [{epoch+1}/{num_epochs}] | LR: {current_lr:.6f}")
    print(f"Train Dice: {train_dice:.4f} | Valid Dice: {valid_dice:.4f}")
    
    # Early stopping check
    if valid_dice > best_valid_dice:
        best_valid_dice = valid_dice
        torch.save(model.state_dict(), 'unet_best.pth')
        print(f">> New best model saved! (Valid Dice: {best_valid_dice:.4f})")
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {patience} epochs with no improvement.")
            break
    
    print("-" * 50)

# Load best model
model.load_state_dict(torch.load('unet_best.pth'))
print(f"\nTraining complete. Best Valid Dice: {best_valid_dice:.4f}")
print("Best model loaded from unet_best.pth")

## 5. Prediction Preview

Visualizing predictions is crucial. We define a utility function to "denormalize" the images (reverting the ImageNet normalization we applied in the dataloader) so they display correctly with standard colors. We then plot the Original Image, Ground Truth Mask, and our model's Predicted Mask side-by-side.


In [5]:
import matplotlib.pyplot as plt

def denormalize(tensor, mean=MEAN, std=STD):
    """Reverse ImageNet normalization for display."""
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    return (tensor.cpu() * std + mean).clamp(0, 1)

def visualize_predictions(model, dataloader, device, num_samples=5):
    model.eval()
    samples_shown = 0
    
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5 * num_samples))
    
    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)
            masks = masks.to(device)
            
            logits = model(images)
            preds = (torch.sigmoid(logits) > 0.5).float()
            
            for i in range(images.size(0)):
                if samples_shown >= num_samples:
                    break
                
                # Denormalize image for correct colors
                img = denormalize(images[i]).permute(1, 2, 0).numpy()
                mask = masks[i].cpu().squeeze().numpy()
                pred = preds[i].cpu().squeeze().numpy()
                
                axes[samples_shown, 0].imshow(img)
                axes[samples_shown, 0].set_title("Original Image")
                axes[samples_shown, 0].axis('off')
                
                axes[samples_shown, 1].imshow(mask, cmap='gray')
                axes[samples_shown, 1].set_title("Ground Truth Mask")
                axes[samples_shown, 1].axis('off')
                
                axes[samples_shown, 2].imshow(pred, cmap='gray')
                axes[samples_shown, 2].set_title("Predicted Mask")
                axes[samples_shown, 2].axis('off')
                
                samples_shown += 1
            
            if samples_shown >= num_samples:
                break
    
    plt.tight_layout()
    plt.show()

# Explicitly reload the best checkpoint before visualizing predictions.
print("Loading best model from unet_best.pth...")
model.load_state_dict(torch.load('unet_best.pth', map_location=device))
model.eval()
print("Best model loaded.\n")

# Visualize 5 samples from the validation set
visualize_predictions(model, valid_loader, device, num_samples=5)

## 6. Best Checkpoint Evaluation

During training, we continually saved the model weights whenever it achieved a new high score on the validation set. We now load that exact `unet_best.pth` checkpoint.

To easily see what the model is predicting, we overlay the predicted mask in **red** directly on top of the original image.


In [6]:
# ── Load the best model explicitly ──
best_model = UNet(in_channels=3, out_channels=1)

# Handle DataParallel saved weights so the checkpoint loads on one or many GPUs.
state_dict = torch.load('unet_best.pth', map_location=device)
# If saved with DataParallel, keys start with 'module.' — strip that prefix
from collections import OrderedDict
new_state_dict = OrderedDict()
for k, v in state_dict.items():
    name = k.replace('module.', '') if k.startswith('module.') else k
    new_state_dict[name] = v
best_model.load_state_dict(new_state_dict)

# Wrap in DataParallel if multiple GPUs
if torch.cuda.device_count() > 1:
    best_model = nn.DataParallel(best_model)
best_model = best_model.to(device)
best_model.eval()
print(f'Best model loaded from unet_best.pth on {device}')

# ── Inference on validation set with flood overlay ──
def test_best_model(model, dataloader, device, num_samples=6):
    model.eval()
    samples_shown = 0
    all_dice = []
    
    fig, axes = plt.subplots(num_samples, 3, figsize=(18, 6 * num_samples))
    
    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)
            masks = masks.to(device)
            
            logits = model(images)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()
            
            # Compute dice for this batch
            batch_dice = dice_coeff(logits, masks)
            all_dice.append(batch_dice)
            
            for i in range(images.size(0)):
                if samples_shown >= num_samples:
                    break
                
                # Denormalize image
                img = denormalize(images[i]).permute(1, 2, 0).numpy()
                mask_gt = masks[i].cpu().squeeze().numpy()
                pred_mask = preds[i].cpu().squeeze().numpy()
                
                # Create a colored overlay where red marks predicted flood pixels.
                overlay = img.copy()
                overlay[pred_mask == 1] = overlay[pred_mask == 1] * 0.4 + np.array([1.0, 0.0, 0.0]) * 0.6
                
                # Per-sample dice
                inter = (pred_mask * mask_gt).sum()
                union = pred_mask.sum() + mask_gt.sum()
                sample_dice = (2 * inter + 1e-6) / (union + 1e-6)
                
                axes[samples_shown, 0].imshow(img)
                axes[samples_shown, 0].set_title('Original Image', fontsize=14)
                axes[samples_shown, 0].axis('off')
                
                axes[samples_shown, 1].imshow(mask_gt, cmap='gray')
                axes[samples_shown, 1].set_title('Ground Truth Mask', fontsize=14)
                axes[samples_shown, 1].axis('off')
                
                axes[samples_shown, 2].imshow(overlay)
                axes[samples_shown, 2].set_title(f'Prediction Overlay (Dice: {sample_dice:.4f})', fontsize=14)
                axes[samples_shown, 2].axis('off')
                
                samples_shown += 1
            
            if samples_shown >= num_samples:
                break
    
    plt.tight_layout()
    plt.show()
    
    avg_dice = sum(all_dice) / len(all_dice)
    print(f'\nOverall Validation Dice (best model): {avg_dice:.4f}')

test_best_model(best_model, valid_loader, device, num_samples=6)